### **1. Import, Load, Clean**

In [1]:
import ast
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

In [2]:
# Load dataset
PATH = "../data/raw/listings.csv"
df = pd.read_csv(PATH)

# Price cleaning
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Percentage cleaning
rates = ["host_response_rate", "host_acceptance_rate"]
for col in rates:
    df[col] = df[col].astype(str).str.replace("%", "").astype(float) / 100

# Remove rows where price is missing
df = df.dropna(subset=["price"]).copy()

In [3]:
cols_to_drop = [

    # identifiers / urls / metadata
    "id",
    "listing_url",
    "scrape_id",
    "source",
    "picture_url",
    "host_id",
    "host_url",
    "host_thumbnail_url",
    "host_picture_url",
    "calendar_updated",
    "calendar_last_scraped",

    # leakage
    "estimated_revenue_l365d",
    "estimated_occupancy_l365d",

    # text fields (no NLP)
    "name",
    "description",
    "neighborhood_overview",
    "host_about",

    # redundant text versions
    "bathrooms_text",
    "host_name",
    "host_verifications",

    # review score redundancy
    "review_scores_accuracy",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_value",

    # availability redundancy
    "availability_60",
    "availability_eoy",

    # review activity redundancy
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",

    # derived night statistics
    "minimum_minimum_nights",
    "maximum_minimum_nights",
    "minimum_maximum_nights",
    "maximum_maximum_nights",
    "minimum_nights_avg_ntm",
    "maximum_nights_avg_ntm",

    # categorical removal from EDA
    "first_review",
    "last_review",
    "license",
    "host_location",
    "host_neighbourhood",
    "neighbourhood",
    "neighbourhood_group_cleansed",
    "has_availability",
    "host_identity_verified",

    # weak categorical predictor
    "host_response_time",
    "host_response_rate"
]

In [4]:
df["price_bin"] = pd.qcut(df["price"], q=5, duplicates="drop")

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["price_bin"]
)

df_train = df_train.drop(columns=["price_bin"])
df_test = df_test.drop(columns=["price_bin"])

In [5]:
df_train["log_price"] = np.log1p(df_train["price"])
df_test["log_price"] = np.log1p(df_test["price"])

In [6]:
X_train = df_train.drop(["price", "log_price"], axis=1)
y_train = df_train["log_price"]

X_test = df_test.drop(["price", "log_price"], axis=1)
y_test = df_test["log_price"]

In [7]:
X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
X_test = X_test.drop(columns=cols_to_drop, errors="ignore")

In [8]:
def parse_amenities(value):
    if pd.isna(value) or value == "":
        return []
    if isinstance(value, list):
        parsed = value
    else:
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    return [str(amenity).strip().strip('"') for amenity in parsed if str(amenity).strip()]

def safe_col(name):
    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace("&", "and")
        .replace(".", "")
    )
    return "".join(ch for ch in safe_name if ch.isalnum() or ch == "_").strip("_")

def fit_selected_amenities(df, amenities_col="amenities", min_share=0.05):
    amenities_lists = df[amenities_col].apply(parse_amenities)
    amenities_counts = pd.Series(
        [amenity for sublist in amenities_lists for amenity in sublist]
    ).value_counts()
    min_count = max(1, int(len(df) * min_share))
    return amenities_counts[amenities_counts >= min_count].index.tolist()

def transform_amenities(df, selected_amenities, amenities_col="amenities"):
    df = df.copy()
    amenities_lists = df[amenities_col].apply(parse_amenities)
    amenities_sets = amenities_lists.apply(set)

    amenity_column_map = {}
    used_cols = set()
    for amenity in selected_amenities:
        base_col = f"has_{safe_col(amenity)}"
        col = base_col
        suffix = 2
        while col in used_cols:
            col = f"{base_col}_{suffix}"
            suffix += 1
        used_cols.add(col)
        amenity_column_map[amenity] = col

    amenity_flags = pd.DataFrame(
        {
            col_name: amenities_sets.apply(lambda s, a=amenity: int(a in s))
            for amenity, col_name in amenity_column_map.items()
        },
        index=df.index,
    )

    amenity_count = pd.Series(amenities_lists.apply(len), name="amenity_count", index=df.index)
    base_df = df.drop(columns=[amenities_col], errors="ignore")

    return pd.concat([base_df, amenity_count, amenity_flags], axis=1)

selected_amenities = fit_selected_amenities(X_train, min_share=0.05)

X_train = transform_amenities(X_train, selected_amenities)
X_test = transform_amenities(X_test, selected_amenities)

print("Selected amenities:", len(selected_amenities))

Selected amenities: 99


In [9]:
X_train["host_acceptance_rate_num"] = pd.to_numeric(
    X_train["host_acceptance_rate"].astype(str).str.replace("%", "", regex=False),
    errors="coerce"
) / 100

X_test["host_acceptance_rate_num"] = pd.to_numeric(
    X_test["host_acceptance_rate"].astype(str).str.replace("%", "", regex=False),
    errors="coerce"
) / 100

X_train = X_train.drop(columns=["host_acceptance_rate"])
X_test = X_test.drop(columns=["host_acceptance_rate"])

In [10]:
def engineer_host_experience(df):
    df = df.copy()

    df["last_scraped"] = pd.to_datetime(df["last_scraped"], errors="coerce")
    df["host_since"] = pd.to_datetime(df["host_since"], errors="coerce")

    reference_date = df["last_scraped"].max()
    df["host_experience_days"] = (reference_date - df["host_since"]).dt.days

    return df.drop(columns=["host_since", "last_scraped"])

X_train = engineer_host_experience(X_train)
X_test = engineer_host_experience(X_test)

In [11]:
print(X_train.shape)
print(X_test.shape)

(4520, 130)
(1131, 130)


In [12]:
X_train.columns

Index(['host_is_superhost', 'host_listings_count', 'host_total_listings_count',
       'host_has_profile_pic', 'neighbourhood_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates',
       ...
       'has_indoor_fireplace', 'has_cleaning_available_during_stay',
       'has_paid_parking_on_premises', 'has_mini_fridge',
       'has_radiant_heating', 'has_keypad',
       'has_pack_n_play_travel_crib___available_upon_request',
       'has_induction_stove', 'host_acceptance_rate_num',
       'host_experience_days'],
      dtype='str', length=130)

In [13]:
pd.set_option('display.max_columns', None)
X_train.describe()

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,availability_30,availability_90,availability_365,number_of_reviews,review_scores_rating,review_scores_cleanliness,review_scores_location,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,amenity_count,has_wifi,has_kitchen,has_hair_dryer,has_hot_water,has_bed_linens,has_hangers,has_dishes_and_silverware,has_essentials,has_tv,has_microwave,has_iron,has_refrigerator,has_fire_extinguisher,has_heating,has_cooking_basics,has_first_aid_kit,has_shampoo,has_washer,has_freezer,has_dishwasher,has_toaster,has_room_darkening_shades,has_dedicated_workspace,has_coffee_maker,has_oven,has_body_soap,has_cleaning_products,has_dining_table,has_drying_rack_for_clothing,has_extra_pillows_and_blankets,has_elevator,has_wine_glasses,has_shower_gel,has_long_term_stays_allowed,has_hot_water_kettle,has_self_check_in,has_smoke_alarm,has_stove,has_free_parking_on_premises,has_host_greets_you,has_crib,has_blender,has_coffee,has_free_street_parking,has_clothing_storage_closet,has_private_entrance,has_private_patio_or_balcony,has_luggage_dropoff_allowed,has_waterfront,has_free_washer__in_unit,has_pets_allowed,has_carbon_monoxide_alarm,has_laundromat_nearby,has_bathtub,has_books_and_reading_material,has_outdoor_furniture,has_portable_fans,has_high_chair,has_outdoor_dining_area,has_clothing_storage,has_central_heating,has_lockbox,has_patio_or_balcony,has_smart_lock,has_ethernet_connection,has_air_conditioning,has_pack_n_play_travel_crib,has_single_level_home,has_bidet,has_board_games,has_lock_on_bedroom_door,has_city_skyline_view,has_crib___available_upon_request,has_coffee_maker_nespresso,has_childrens_books_and_toys,has_paid_parking_off_premises,has_bbq_grill,has_dryer,has_exterior_security_cameras_on_property,has_shared_beach_access,has_mountain_view,has_baking_sheet,has_backyard,has_free_dryer__in_unit,has_beach_access,has_conditioner,has_smoking_allowed,has_childrens_dinnerware,has_pocket_wifi,has_paid_parking_garage_off_premises,has_garden_view,has_indoor_fireplace,has_cleaning_available_during_stay,has_paid_parking_on_premises,has_mini_fridge,has_radiant_heating,has_keypad,has_pack_n_play_travel_crib___available_upon_request,has_induction_stove,host_acceptance_rate_num,host_experience_days
count,4518.000000,4518.000000,4520.000000,4520.000000,4520.000000,4516.000000,4518.000000,4509.000000,4520.000000,4.520000e+03,4520.000000,4520.000000,4520.000000,4520.000000,4098.000000,4098.000000,4098.000000,4520.000000,4520.000000,4520.000000,4520.000000,4098.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4520.000000,4302.000000,4518.000000
mean,21.817176,30.585215,43.260544,-2.5

### **2. Handling missing values**

In [14]:
missing = X_train.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0]

review_scores_location       9.336283
reviews_per_month            9.336283
review_scores_cleanliness    9.336283
review_scores_rating         9.336283
host_acceptance_rate_num     4.823009
host_is_superhost            2.898230
beds                         0.243363
bathrooms                    0.088496
host_listings_count          0.044248
host_total_listings_count    0.044248
bedrooms                     0.044248
host_has_profile_pic         0.044248
host_experience_days         0.044248
dtype: float64

In [15]:
num_cols = X_train.select_dtypes(include="number").columns

num_imputer = SimpleImputer(strategy="median")

X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

In [16]:
cat_cols = X_train.select_dtypes(include="object").columns

cat_imputer = SimpleImputer(strategy="most_frequent")

X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/1189760543.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns


In [17]:
X_train.isna().sum().sum(), X_test.isna().sum().sum()

(np.int64(0), np.int64(0))

### **3. Transformations**

In [18]:
BILBAO_LAT = 43.2630
BILBAO_LON = -2.9350

def distance_to_bilbao(lat, lon):
    return np.sqrt((lat - BILBAO_LAT)**2 + (lon - BILBAO_LON)**2)

X_train["distance_to_bilbao"] = distance_to_bilbao(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_bilbao"] = distance_to_bilbao(
    X_test["latitude"], X_test["longitude"]
)

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/3943955730.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train["distance_to_bilbao"] = distance_to_bilbao(
/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/3943955730.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test["distance_to_bilbao"] = distance_to_bilbao(


In [19]:
DONOSTIA_LAT = 43.3183
DONOSTIA_LON = -1.9812

def distance_to_donostia(lat, lon):
    return np.sqrt((lat - DONOSTIA_LAT)**2 + (lon - DONOSTIA_LON)**2)

X_train["distance_to_donostia"] = distance_to_donostia(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_donostia"] = distance_to_donostia(
    X_test["latitude"], X_test["longitude"]
)

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/2074891443.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train["distance_to_donostia"] = distance_to_donostia(
/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/2074891443.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test["distance_to_donostia"] = distance_to_donostia(


In [20]:
VITORIA_LAT = 42.8467
VITORIA_LON = -2.6726

def distance_to_vitoria(lat, lon):
    return np.sqrt((lat - VITORIA_LAT)**2 + (lon - VITORIA_LON)**2)

X_train["distance_to_vitoria"] = distance_to_vitoria(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_vitoria"] = distance_to_vitoria(
    X_test["latitude"], X_test["longitude"]
)

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/2885989634.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train["distance_to_vitoria"] = distance_to_vitoria(
/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/2885989634.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test["distance_to_vitoria"] = distance_to_vitoria(


In [21]:
COAST_LAT = 43.3623
COAST_LON = -3.0136

def distance_to_coast(lat, lon):
    return np.sqrt((lat - COAST_LAT)**2 + (lon - COAST_LON)**2)

X_train["distance_to_coast"] = distance_to_coast(
    X_train["latitude"], X_train["longitude"]
)

X_test["distance_to_coast"] = distance_to_coast(
    X_test["latitude"], X_test["longitude"]
)

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/2711187647.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train["distance_to_coast"] = distance_to_coast(
/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/2711187647.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test["distance_to_coast"] = distance_to_coast(


In [22]:
clip_cols = [
    "beds",
    "bathrooms",
    "bedrooms",
    "minimum_nights",
    "maximum_nights",
    "host_acceptance_rate_num"
]

for col in clip_cols:
    
    lower = X_train[col].quantile(0.01)
    upper = X_train[col].quantile(0.99)

    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

In [23]:
skewed = [
    "minimum_nights",
    "accommodates",
    "maximum_nights",
    "number_of_reviews",
    "reviews_per_month"
]

for col in skewed:
    X_train[col] = np.log1p(X_train[col])
    X_test[col] = np.log1p(X_test[col])

In [24]:
cat_cols = X_train.select_dtypes(include="object").columns
cat_cols

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/1847301736.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns


Index(['host_is_superhost', 'host_has_profile_pic', 'neighbourhood_cleansed',
       'property_type', 'room_type', 'instant_bookable'],
      dtype='str')

In [25]:
X_train["instant_bookable"] = X_train["instant_bookable"].map({"t": 1, "f": 0})
X_test["instant_bookable"] = X_test["instant_bookable"].map({"t": 1, "f": 0})

In [26]:
TOP_K = 10

top_properties = (
    X_train["property_type"]
    .value_counts()
    .nlargest(TOP_K)
    .index
)

X_train["property_type_clean"] = X_train["property_type"].where(
    X_train["property_type"].isin(top_properties),
    "Other"
)

X_test["property_type_clean"] = X_test["property_type"].where(
    X_test["property_type"].isin(top_properties),
    "Other"
)

X_train = X_train.drop(columns=["property_type"])
X_test = X_test.drop(columns=["property_type"])

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/3970232632.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train["property_type_clean"] = X_train["property_type"].where(
/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/3970232632.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test["property_type_clean"] = X_test["property_type"].where(


In [27]:
TOP_K = 10

top_neighbourhoods = (
    X_train["neighbourhood_cleansed"]
    .value_counts()
    .nlargest(TOP_K)
    .index
)

X_train["neighbourhood_cleansed_clean"] = X_train["neighbourhood_cleansed"].where(
    X_train["neighbourhood_cleansed"].isin(top_neighbourhoods),
    "Other"
)

X_test["neighbourhood_cleansed_clean"] = X_test["neighbourhood_cleansed"].where(
    X_test["neighbourhood_cleansed"].isin(top_neighbourhoods),
    "Other"
)

X_train = X_train.drop(columns=["neighbourhood_cleansed"])
X_test = X_test.drop(columns=["neighbourhood_cleansed"])

/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/1675569974.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train["neighbourhood_cleansed_clean"] = X_train["neighbourhood_cleansed"].where(
/var/folders/cy/z3d245cd40l5b67ymvkjkht40000gn/T/ipykernel_7048/1675569974.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test["neighbourhood_cleansed_clean"] = X_test["neighbourhood_cleansed"].where(


In [28]:
categorical_cols = [
    "property_type_clean",
    "room_type",
    "neighbourhood_cleansed_clean",
    "host_is_superhost",
    "host_has_profile_pic"
]

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

### **4. Model**

#### **4.a. Ridge Regression** 

In [29]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [30]:
alphas = np.logspace(-3, 3, 50)

ridge_cv = RidgeCV(
    alphas=alphas,
    scoring="neg_root_mean_squared_error",
    cv=5
)

ridge_cv.fit(X_train_scaled, y_train)

print("Best alpha:", ridge_cv.alpha_)

Best alpha: 25.595479226995334


In [31]:
ridge = Ridge(alpha=ridge_cv.alpha_)

ridge.fit(X_train_scaled, y_train)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",np.float64(25.595479226995334)
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochasti

In [32]:
scores = cross_val_score(
    ridge,
    X_train_scaled,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)

rmse = -scores.mean()

print("CV RMSE:", rmse)

CV RMSE: 0.4680706446395515


#### **4.b. Random Forest**

In [33]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_scores = cross_val_score(
    rf,
    X_train,   # unscaled
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)

rf_rmse = -rf_scores.mean()

print("Random Forest CV RMSE:", rf_rmse)

Random Forest CV RMSE: 0.40928023420990095


#### **4.c. Gradient Boosting**

In [34]:
hgb = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=300,
    max_depth=6,
    min_samples_leaf=20,
    l2_regularization=0.0,
    random_state=42
)

hgb_scores = cross_val_score(
    hgb,
    X_train,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)

hgb_rmse = -hgb_scores.mean()

print("Gradient Boosting CV RMSE:", hgb_rmse)

Gradient Boosting CV RMSE: 0.3795303576981818
